# IMPORT MODULES, INSTANTIATE CLASS

In [1]:
import glob
import os

import pandas as pd

from geoai.utils_geo.raster_ops import RasterOperations
from geoai.utils_ds.dataframe_ops import DataFrameOperations
from geoai.utils_geo.vector_ops import VectorOperations

raster_ops = RasterOperations()
dataframe_ops = DataFrameOperations()
vector_ops = VectorOperations()

# SET GLOBAL VAR

In [2]:
RASTER_PATH = r"raster_files\QC_2018_2023.tif"
N_BANDS = 5
BAND_NAMES = ["BLUE", "GREEN", "RED", "NIR", "SWIR"]

# CLIP ROI SHAPEFILE TO RASTER

In [3]:
# list the shapefiles in the shapefiles folder
shapefile_list = glob.glob(os.path.join("shapefiles", "*.shp"))

for shapefile in shapefile_list: # Loop through all shapefiles in the shapefiles folder
    roi_name = os.path.splitext(os.path.basename(shapefile))[0] # Get the name of the shapefile without the extension
    output_raster_path = os.path.join("raster_files", f"{roi_name}.tif") # Define the output raster path 
    vector_ops.clip_raster_with_shapefile(RASTER_PATH, shapefile, output_raster_path) # Clip the raster with the shapefile
    print(f"Done clipping {roi_name}")

Done clipping builtup
Done clipping grass
Done clipping road
Done clipping trees


# MAKE A DF ON EVERY ROI RASTER

In [4]:
list_of_roi = ["builtup", "trees", "grass", "road"]

# make a list of all the raster files in the raster_files folder
raster_files = glob.glob(os.path.join("raster_files", "*.tif"))

for roi_path in raster_files:
    roi_name = os.path.splitext(os.path.basename(roi_path))[0] # Get the name of the raster file without the extension
    if roi_name in list_of_roi:
        df_bands = [] # Create an empty list to store the dataframes
        array = raster_ops.raster_to_array(roi_path) # Convert the raster to an array
        for band_index, band_name in zip(range(N_BANDS), BAND_NAMES): # Loop through the bands
            flat = raster_ops.flatten_array(array, band_index) # Flatten the array
            df = dataframe_ops.convert_to_df(flat, band_name) # Convert the array to a dataframe
            df = df.loc[~(df==0).all(axis=1)] # remove rows if all of its column is zero
            df_bands.append(df) # Append the dataframe to the df_bands list
        final_df_per_bands = pd.concat(df_bands, axis=1) # Concatenate the dataframes in the df_bands list 
        final_df_per_bands["Landcover"] = roi_name # Add a column to the dataframe with the name of the landcover
        final_df_per_bands.to_csv(f"csv_files\{roi_name}.csv", index = False) # Save the dataframe to a csv file
        print(final_df_per_bands.head(1))
        print()

           BLUE   GREEN     RED          NIR     SWIR Landcover
483  835.333313  920.75  1301.0  1478.666626  1976.75   builtup

      BLUE        GREEN    RED          NIR    SWIR Landcover
163  700.0  1049.333374  883.5  3401.333252  2666.0     grass

       BLUE        GREEN          RED     NIR    SWIR Landcover
438  1030.0  1094.857178  1135.333374  1197.5  1720.5      road

          BLUE   GREEN     RED     NIR    SWIR Landcover
53  403.916656  553.25  394.75  2806.5  1919.5     trees



# CREATE 1 CSV THAT OF ALL ROIs DATAFRAMES

In [5]:
# make a list of all the csv files in the csv_files folder
roi_csv_files = glob.glob(os.path.join("csv_files", "*.csv"))

# Create an empty list to store the dataframes
df_roi = []


for roi_csv_path in roi_csv_files: # Loop through all the csv files in the csv_files folder
    roi_csv_name = os.path.splitext(os.path.basename(roi_csv_path))[0] # Get the name of the csv file without the extension
    if roi_csv_name in list_of_roi:
        df = pd.read_csv(roi_csv_path) # Read the csv file
        df_roi.append(df) # Append the dataframe to the df_roi list
final_training_data = pd.concat(df_roi, axis=0) # Concatenate the dataframes in the df_roi list
final_training_data.to_csv("csv_files/dataset.csv", index=False) # Save the dataframe to a csv file

END